# RetentionAI — 05. Leakage-Safe Pipeline (Stage 6)

**Objective:** turn Stages 3-5's decisions into one shared, leakage-safe
pipeline. Concretely, this notebook resolves three things that were
deliberately deferred until now:

1. The `SeniorCitizen` 0/1 vs. `Yes`/`No` inconsistency (flagged Stage 3).
2. The `tenure` / `TotalCharges` / `MonthlyCharges` multicollinearity flag
   (flagged Stage 4's heatmap, explicitly not acted on there).
3. An encoding scheme for every remaining categorical column.

**Rule for this notebook:** split into train/test *first*. Every fitted
step — the IV filter, one-hot category levels, VIF-based drops, the
scaler — is fit on training data only, then applied unchanged to the test
set. Fitting any of this on the full dataset before splitting would leak
test-set statistics into decisions the model shouldn't have access to yet.

> Runs on synthetic data in this sandbox — re-run unchanged on the real
> `data/raw/telco_churn.csv` (no code changes needed).

In [1]:
import pandas as pd

from src.config import RAW_CSV_PATH, PROCESSED_CSV_PATH
from src.features.pipeline import prepare_features, run_stage6_split

df = pd.read_csv(RAW_CSV_PATH)
print("Raw shape:", df.shape)
df.head(3)

Raw shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 1. Deterministic prep — safe before any split exists

`prepare_features()` does only row-by-row, non-fitted work: carrying
forward the Stage 3/4 cleaning, rebuilding the one feature that survived
Stage 5 (`ContractCommitmentMonths` — `TotalAddOnServices`,
`IsNewCustomer`, and `AvgMonthlySpend` were all rejected in `ADR-006` and
are deliberately not rebuilt here), and resolving the `SeniorCitizen`
inconsistency by mapping every other true binary column to match its
existing 0/1 convention.

This is also exactly what gets written to `PROCESSED_CSV_PATH` — cleaned
and engineered, but **not yet** IV/VIF-filtered or scaled. Those steps
depend on a specific train split and have to be re-fit at model-training
time; baking them into a static file would quietly freeze one particular
split's decisions as if they applied universally.

In [2]:
from src.config import PROJECT_ROOT

X, y = prepare_features(df)

print("Shape:", X.shape)
print("Any nulls:", X.isna().sum().sum())
print("Columns:", X.columns.tolist())

PROCESSED_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
processed = X.copy()
processed["Churn_numeric"] = y
processed.to_csv(PROCESSED_CSV_PATH, index=False)
print(f"\nWrote {processed.shape} to {(PROCESSED_CSV_PATH).relative_to(PROJECT_ROOT)}")

Shape: (7043, 19)
Any nulls: 0
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'ContractCommitmentMonths']

Wrote (7043, 20) to data\processed\telco_churn_cleaned.csv


## 2. Structural redundancy, resolved by construction, not by VIF discovery

`Contract` is dropped inside `prepare_features()` — before this notebook
even runs `train_test_split`. `ContractCommitmentMonths` is an exact
1-to-1 relabeling of `Contract`, so keeping both would hand VIF a
perfectly singular pair (infinite VIF), which isn't a redundancy worth
letting an iterative algorithm discover one column at a time — it's true
by definition, and reasoning about it directly is clearer than waiting
for a number to confirm it.

## 3. Split, then fit everything stateful on train only

`run_stage6_split()` does: `prepare_features` → `train_test_split` →
IV filter (whole categorical columns, before encoding — keeping or
dropping `PaymentMethod` is a decision about the whole concept, not
individual category levels) → one-hot encode survivors → VIF filter on
the encoded matrix → scale. Every fitted step only ever sees `X_train`.

In [3]:
X_train, X_test, y_train, y_test, artifacts = run_stage6_split(df)

print("=== IV table (train only) ===")
print(artifacts["iv_table"])
print("\nIV-dropped (<0.02):", [f for f in artifacts["iv_table"]["feature"]
                                   if f not in artifacts["iv_keep"]])

=== IV table (train only) ===
                     feature      iv                              strength
0   ContractCommitmentMonths  1.2357  suspiciously strong -- check leakage
1             OnlineSecurity  0.7325  suspiciously strong -- check leakage
2                     tenure  0.7184  suspiciously strong -- check leakage
3                TechSupport  0.7076  suspiciously strong -- check leakage
4            InternetService  0.6328  suspiciously strong -- check leakage
5               OnlineBackup  0.5357  suspiciously strong -- check leakage
6           DeviceProtection  0.4972                                strong
7              PaymentMethod  0.4806                                strong
8            StreamingMovies  0.3852                                strong
9                StreamingTV  0.3825                                strong
10            MonthlyCharges  0.3080                                strong
11              TotalCharges  0.2646                                me

## 4. VIF — resolving the deferred tenure/TotalCharges/MonthlyCharges flag

Watch for two things in the output below:
1. Several `"...No internet service"` dummy columns are likely to get
   dropped first — that category is structurally identical across
   `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`,
   `StreamingTV`, and `StreamingMovies` (a customer with no internet has
   `"No internet service"` in all six simultaneously), so those dummies
   are close to perfectly collinear with each other. This is VIF catching
   a real structural fact about the dataset, not a bug.
2. Whether `tenure`, `MonthlyCharges`, and `TotalCharges` finally get
   resolved — this is the flag from the Stage 4 heatmap, explicitly
   deferred until here.

In [4]:
print("VIF-dropped, in order:")
for col in artifacts["vif_dropped"]:
    print(" -", col)

print("\n=== Final VIF table (post-filter) ===")
print(artifacts["final_vif_table"])

VIF-dropped, in order:
 - InternetService_No
 - DeviceProtection_No internet service
 - TechSupport_No internet service
 - OnlineBackup_No internet service
 - OnlineSecurity_No internet service
 - StreamingTV_No internet service
 - MonthlyCharges
 - TotalCharges

=== Final VIF table (post-filter) ===
                                  feature       VIF
0                                  tenure  6.980622
1                ContractCommitmentMonths  4.190667
2                                 Partner  2.789630
3                     StreamingMovies_Yes  2.690592
4                         StreamingTV_Yes  2.669412
5             InternetService_Fiber optic  2.655605
6                        PaperlessBilling  2.622182
7     StreamingMovies_No internet service  2.483276
8                    DeviceProtection_Yes  2.261895
9                        OnlineBackup_Yes  2.111159
10         PaymentMethod_Electronic check  2.077652
11                        TechSupport_Yes  2.041191
12                    

**Verdict:** *(fill in once run on real data)* — which of `tenure` /
`MonthlyCharges` / `TotalCharges` survived, and does the survivor match
the one with the highest standalone IV from the table above? If VIF and
IV point to the same column, that's corroborating evidence, not a
coincidence — the column carrying the most real signal is also the one
best worth keeping when a redundant group has to be trimmed.

## 5. Final shapes — no resampling applied

No SMOTE, no class-weighting, no calibration holdout carved out here.
None of those have an earned reason yet in this project's ADR history —
adding them now, just because they're standard technique, would repeat
exactly the mistake `ADR-000` describes: reaching for advanced-sounding
tools before the stage that would actually justify them. Whether class
imbalance needs addressing — and if so, resampling vs. `class_weight`
vs. neither — is a Stage 7/8 modeling decision, made once there's an
actual baseline result showing whether it's a real problem.

In [5]:
print("X_train:", X_train.shape, "| churn rate:", round(y_train.mean(), 3))
print("X_test: ", X_test.shape, "| churn rate:", round(y_test.mean(), 3))
print("\nFinal feature columns going into Stage 7:")
print(X_train.columns.tolist())

X_train: (5634, 17) | churn rate: 0.265
X_test:  (1409, 17) | churn rate: 0.265

Final feature columns going into Stage 7:
['ContractCommitmentMonths', 'tenure', 'PaperlessBilling', 'Dependents', 'Partner', 'SeniorCitizen', 'InternetService_Fiber optic', 'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
